# 02 — Narrative Inspection

Load generated narratives from the DB, read them alongside their ground-truth SHAP values,
and manually inspect quality before running automated evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv

load_dotenv('../.env')

from src.config import load_config
from src.db import db_connection, get_narratives_for_run, list_runs

cfg = load_config('../config/default.yaml')

In [ ]:
# List all available runs
with db_connection(cfg.storage.db_path) as conn:
    runs = list_runs(conn)

pd.DataFrame(runs)

In [ ]:
# Set the run_id you want to inspect
RUN_ID = runs[0]['run_id'] if runs else None
print('Inspecting run:', RUN_ID)

In [ ]:
with db_connection(cfg.storage.db_path) as conn:
    narratives = get_narratives_for_run(conn, RUN_ID)

narr_df = pd.DataFrame(narratives)
print(f'{len(narr_df)} narratives loaded')
narr_df.head()

In [ ]:
# Summary: narratives per model × strategy
narr_df.groupby(['model_id', 'prompt_strategy']).size().unstack(fill_value=0)

In [ ]:
# Inspect a random narrative alongside its SHAP context
import pandas as pd as pd2  # avoid re-import warning
from src.data_loader import format_shap_table

sample = narr_df.sample(1).iloc[0]
dataset_cfg = cfg.get_dataset(sample['dataset'])
raw_df = pd.read_csv(f'../{dataset_cfg.path}')
row = raw_df.iloc[sample['instance_id']]

print('=== SHAP VALUES ===')
print(format_shap_table(row, dataset_cfg.shap_col_prefix))
print()
print(f'=== NARRATIVE ({sample["model_id"]} | {sample["prompt_strategy"]}) ===')
print(sample['narrative_text'])

In [ ]:
# Browse narratives for a specific model and strategy
MODEL = 'claude-opus'
STRATEGY = 'zero_shot'
DATASET = 'adult'
N = 5  # how many to show

subset = narr_df[
    (narr_df['model_id'] == MODEL) &
    (narr_df['prompt_strategy'] == STRATEGY) &
    (narr_df['dataset'] == DATASET)
].head(N)

for _, row in subset.iterrows():
    print(f'--- Instance {row["instance_id"]} ---')
    print(row['narrative_text'])
    print()